In [1]:
!pwd

/home/hgkahng/Workspaces/soft-prompt/notebooks/subj


In [2]:
import os
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
root_dir = Path("../../").resolve()
print(root_dir)

/home/hgkahng/Workspaces/soft-prompt


In [4]:
data_dir = root_dir / "data/subj"
os.makedirs(data_dir, exist_ok=True)

In [ ]:
from datasets import load_dataset

ds = load_dataset("SetFit/subj", trust_remote_code=True)

In [7]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2000
    })
})

In [8]:
df_train = ds['train'].to_pandas()
df_test = ds['test'].to_pandas()

In [9]:
for i, row in df_test.iterrows():
    print(f"(i={i:>4}, y={row['label']}) {row['text']}")
    if i == 10:
        break

(i=   0, y=0) pirates of the caribbean is a sweeping action-adventure story set in an era when villainous pirates scavenged the caribbean seas .
(i=   1, y=1) my 5-year-old niece emily was so thrilled by the first 20 minutes , she announced , `` i 'm gon na try to stay awake through the whole movie ! `` , which she then proceeded to do .
(i=   2, y=1) although dampened by intermittent preachiness and an unconvincingly pat and uplifting resolution . . . changing lanes nevertheless taps into emotions so convincing it elevates the movie above its own shortcomings .
(i=   3, y=0) jimmy ray is an alcoholic , still consumed by his grief and angry with his sons for wanting to play music .
(i=   4, y=0) bruce soon learns that being god is very challenging .
(i=   5, y=0) he confesses to committing a series of particularly horrifying murders of exclusively female victims .
(i=   6, y=1) this is a startling film that gives you a fascinating , albeit depressing view of iranian rural life close to

# Compute Embeddings
- Model: `text-embedding-3-small`

In [10]:
import asyncio

from langchain_openai import OpenAIEmbeddings
from typing import List


MODEL: str = 'text-embedding-3-small'


async def batch_embed_documents(
    texts: List[str], batch_size: int = 10):
    
    # Initialize the embeddings class
    embedder = OpenAIEmbeddings(model=MODEL,
                                max_retries=5,
                                request_timeout=120.,)

    # Create batches of texts
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    print(f">> Split {len(texts)} documents into {len(batches)} batches")
    
    embeddings = []
    
    # Process each batch
    for i, batch in enumerate(batches):
        print(f">> Processing batch {i+1:>4,}/{len(batches):>4,} ({len(batch)} documents)")
        
        # Get embeddings for the current batch
        batch_embeddings = await embedder.aembed_documents(batch)
        embeddings.extend(batch_embeddings)
        
        print(f">>  Completed batch {i+1:>4,}/{len(batches):>4,}")
        
        # Optional: Add a delay between batches to avoid rate limits
        if i < len(batches) - 1:
            await asyncio.sleep(0.5)  # 500ms delay between batches
    
    print(f">> Generated {len(embeddings):,} embeddings in total")

    return np.array(embeddings)

In [11]:
df_train.sample(10)

,text,label,label_text
2784,"however due to roxie 's new found fame , velma...",0,objective
6503,"sadly , 'garth ' has n't progressed as nicely ...",1,subjective
1774,. . . liotta is put in an impossible spot beca...,1,subjective
2473,bernal 's transformation from naive priest to ...,1,subjective
3399,[ a ] smarter and much funnier version of the ...,1,subjective
7829,"in its chicken heart , crush goes to absurd le...",1,subjective
3279,insomnia is one of the year 's best films and ...,1,subjective
1040,a shrewd and splendidly volatile b movie struc...,1,subjective
6100,a compelling motion picture that illustrates a...,1,subjective
363,no ten & # 237 ; a que enfrentarse con la estr...,0,objective


In [ ]:
X_train = await batch_embed_documents(df_train['text'], batch_size=100)

In [18]:
X_train.shape

(8000, 1536)

In [30]:
y_train = df_train['label'].values
print(y_train.shape)

(8000,)


In [15]:
df_test.sample(10)

,text,label,label_text
1581,"while fighting to save themselves , they must ...",0,objective
726,craig bartlett and director tuck tucker should...,1,subjective
1859,"without resorting to camp or parody , haynes (...",1,subjective
1828,"soon , he finds that his fellow patients are b...",0,objective
542,"doctors are n't supposed to play god , but som...",0,objective
1616,worth a look by those on both sides of the iss...,1,subjective
1891,my wife is an actress is an utterly charming f...,1,subjective
580,a captivating and intimate study about dying a...,1,subjective
56,at the height of hitler 's infamous u-boat war...,0,objective
1913,the egoists is an ensemble piece that reveals ...,0,objective


In [ ]:
X_test = await batch_embed_documents(df_test['text'], batch_size=100)

In [19]:
X_test.shape

(2000, 1536)

In [29]:
y_test = df_test['label'].values
print(y_test.shape)

(2000,)


Save embeddings

In [23]:
emb_save_dir = data_dir / f"embeddings/openai/{MODEL}"
os.makedirs(emb_save_dir, exist_ok=True)
print("Embeddings will be saved to:\n", emb_save_dir)

Embeddings will be saved to:
 /home/hgkahng/Workspaces/soft-prompt/data/subj/embeddings/openai/text-embedding-3-small


In [31]:
np.save(emb_save_dir / "train.features.npy", X_train)
np.save(emb_save_dir / "train.labels.npy", y_train)
np.save(emb_save_dir / "test.features.npy", X_test)
np.save(emb_save_dir / "test.labels.npy", y_test)